In [1]:
import re
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("/Users/apple/Desktop/UCL/skills/python3")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
CLEANED_DIR = PROJECT_ROOT / "data" / "cleaned"

CLEANED_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
landform_df = pd.read_csv(RAW_DIR / "wikipedia_landforms.csv")
climate_df = pd.read_csv(RAW_DIR / "wikipedia_climate.csv")
gutenberg_df = pd.read_csv(RAW_DIR / "gutenberg_nature.csv")

print("landform raw:", len(landform_df))
print("climate raw:", len(climate_df))
print("gutenberg raw:", len(gutenberg_df))

landform raw: 532
climate raw: 359
gutenberg raw: 904


In [3]:
def normalize_text(text):
    if pd.isna(text):
        return ""
    
    text = str(text)

    # 去掉 Wikipedia 引用编号 [1] [23]
    text = re.sub(r"\[\d+\]", "", text)

    # 去掉网址
    text = re.sub(r"http\S+|www\S+", "", text)

    # 去掉多余空格和换行
    text = re.sub(r"\s+", " ", text).strip()

    # 去掉首尾引号和奇怪符号
    text = text.strip(" \"'“”‘’")

    return text

In [4]:
for df in [landform_df, climate_df, gutenberg_df]:
    df["cleaned_text"] = df["raw_text"].apply(normalize_text)
    df["word_count"] = df["cleaned_text"].apply(lambda x: len(str(x).split()))

In [5]:
def post_filter(df, min_words=40, max_words=250):
    df = df.copy()
    df = df[(df["word_count"] >= min_words) & (df["word_count"] <= max_words)]
    df = df.drop_duplicates(subset=["cleaned_text"])
    df = df.reset_index(drop=True)
    return df

landform_clean = post_filter(landform_df)
climate_clean = post_filter(climate_df)
gutenberg_clean = post_filter(gutenberg_df)

In [6]:
print("landform clean:", len(landform_clean))
print("climate clean:", len(climate_clean))
print("gutenberg clean:", len(gutenberg_clean))

landform clean: 532
climate clean: 358
gutenberg clean: 903


In [7]:
landform_clean[["page_title", "cleaned_text", "word_count"]].sample(5)

,page_title,cleaned_text,word_count
269,Coral reef,"Patch reef – common, isolated, comparatively s...",61
511,Forest,Tropical dry forests are characteristic of are...,110
62,Desert,"Many centuries later, both world wars saw figh...",70
495,Forest,The first known forests on Earth arose in the ...,119
479,Tundra,Alpine tundra occurs in mountains worldwide. T...,56


In [8]:
climate_clean[["page_title", "cleaned_text", "word_count"]].sample(5)

,page_title,cleaned_text,word_count
240,Atmosphere,"Aside from Mercury, all Solar System planets h...",86
183,Permafrost,"Permafrost is soil, rock or sediment that is f...",191
232,Permafrost,Between the middle of the 19th century and the...,134
165,Floodplain,Crevasses are formed by breakout events from t...,50
209,Permafrost,Permafrost thaw is associated with a wide rang...,51


In [9]:
gutenberg_clean[["page_title", "cleaned_text", "word_count"]].sample(5)

,page_title,cleaned_text,word_count
311,the_yosemite,Passing the Cathedral we descend into the deli...,71
546,thousand_mile_walk,"It was not, therefore, a new species of advent...",201
385,the_yosemite,In Oregon and Washington it forms immense fore...,138
697,thousand_mile_walk,"By this time I was becoming faint, and in maki...",75
109,canoeing_in_the_wilderness,It was unusual for the woods to be so distant ...,54


In [10]:
landform_clean.to_csv(CLEANED_DIR / "landforms_clean.csv", index=False)
climate_clean.to_csv(CLEANED_DIR / "climate_clean.csv", index=False)
gutenberg_clean.to_csv(CLEANED_DIR / "gutenberg_clean.csv", index=False)

print("Saved:")
print(CLEANED_DIR / "landforms_clean.csv")
print(CLEANED_DIR / "climate_clean.csv")
print(CLEANED_DIR / "gutenberg_clean.csv")

Saved:
/Users/apple/Desktop/UCL/skills/python3/data/cleaned/landforms_clean.csv
/Users/apple/Desktop/UCL/skills/python3/data/cleaned/climate_clean.csv
/Users/apple/Desktop/UCL/skills/python3/data/cleaned/gutenberg_clean.csv


In [11]:
summary_df = pd.DataFrame({
    "dataset": ["landforms", "climate", "gutenberg"],
    "rows_after_cleaning": [
        len(landform_clean),
        len(climate_clean),
        len(gutenberg_clean)
    ],
    "avg_word_count": [
        landform_clean["word_count"].mean(),
        climate_clean["word_count"].mean(),
        gutenberg_clean["word_count"].mean()
    ]
})

summary_df

,dataset,rows_after_cleaning,avg_word_count
0,landforms,532,94.654135
1,climate,358,94.326816
2,gutenberg,903,111.049834


In [12]:
summary_df.to_csv(CLEANED_DIR / "dataset_summary.csv", index=False)